In [1]:
!pip install langchain_community

In [2]:
!pip install langchain langchain-pinecone langchain-groq

In [3]:
!pip install langchain-huggingface

  Using cached langchain_huggingface-1.2.2-py3-none-any.whl.metadata (4.0 kB)
Using cached langchain_huggingface-1.2.2-py3-none-any.whl (31 kB)


In [4]:
from google.colab import userdata
pinecone_api_key=userdata.get("pinecone_api_key")

In [5]:
from pinecone import Pinecone
from langchain_huggingface import HuggingFaceEmbeddings
pc=Pinecone(api_key=pinecone_api_key)

In [14]:
embeddings=HuggingFaceEmbeddings(model="BAAI/bge-large-en-v1.5")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [7]:
text="""Artificial Intelligence (AI) is transforming the world at an unprecedented pace, impacting industries, education, healthcare, business, entertainment, and daily life. AI refers to the simulation of human intelligence in machines that are programmed to think, learn, and solve problems. Over the last decade, advancements in machine learning, deep learning, natural language processing, and computer vision have significantly improved the capabilities of intelligent systems. These technologies allow computers to analyze massive amounts of data, recognize patterns, make predictions, and even generate human-like text and images.

Machine learning is one of the most important branches of AI. It enables systems to learn from data without being explicitly programmed. Algorithms are trained using datasets, and the model improves its performance over time. Supervised learning, unsupervised learning, and reinforcement learning are the three primary categories of machine learning. Supervised learning uses labeled data to train models, while unsupervised learning identifies hidden patterns in unlabeled data. Reinforcement learning focuses on decision-making by rewarding successful actions and penalizing incorrect ones.

Deep learning, a subset of machine learning, uses neural networks inspired by the human brain. These networks contain multiple layers that process information in increasingly abstract ways. Deep learning models are widely used in applications such as image recognition, speech processing, autonomous vehicles, and recommendation systems. Popular frameworks like TensorFlow and PyTorch have made it easier for developers and researchers to build and train deep learning models efficiently.

Natural Language Processing (NLP) is another major field of AI that focuses on enabling computers to understand and generate human language. NLP powers chatbots, virtual assistants, translation systems, and search engines. Large Language Models (LLMs) such as GPT models have revolutionized the NLP field by generating highly coherent and context-aware responses. These models are trained on enormous amounts of text data and can perform tasks such as summarization, question answering, content generation, and sentiment analysis.

Retrieval-Augmented Generation (RAG) is an advanced AI architecture that combines information retrieval with language generation. Instead of relying solely on the knowledge stored in a model’s parameters, RAG systems retrieve relevant information from external sources such as vector databases, documents, or knowledge bases before generating a response. This approach improves accuracy, reduces hallucinations, and allows AI systems to access updated information dynamically. Vector databases like Pinecone, Chroma, FAISS, and Astra DB are commonly used in RAG applications to store and retrieve embeddings efficiently."""

In [8]:
from langchain_core.documents import Document
document=Document(page_content=text)

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter=RecursiveCharacterTextSplitter(chunk_size=100,chunk_overlap=10,length_function=len,separators=[" "])
chunks=splitter.split_documents([document])

In [10]:
print(len(chunks))

32


In [15]:
from pinecone import ServerlessSpec
index_name="rag"
if not pc.has_index(index_name):
  pc.create_index(
    name=index_name,
    dimension=1024,
    metric="cosine",
    sc=ServerlessSpec(cloude="aws",region="us-east-1")
  )
index=pc.Index(index_name)

In [16]:
from langchain_pinecone import PineconeVectorStore
vector=PineconeVectorStore(index=index,embedding=embeddings)

In [17]:
vectorstore=vector.add_documents(documents=chunks)

In [19]:
print(len(vectorstore))

32


In [21]:
results = vector.similarity_search(
    "LangChain provides abstractions to make working with LLMs easy",
    k=2,)

In [25]:
print(results)

[Document(id='82cbb45b-646b-47ee-9804-10c14bb559d0', metadata={}, page_content='virtual assistants, translation systems, and search engines. Large Language Models (LLMs) such as'), Document(id='3cd02c8c-d635-43c8-9168-fff23d6125b7', metadata={}, page_content='on enabling computers to understand and generate human language. NLP powers chatbots, virtual')]
